## Sequence Purchasing Planning/Brainstorming

Lead     : `<Alex / AlexLeonardos>`

Issue    : [Github Issue #84](https://github.com/petadex/igem-toronto/issues/) — _Algorithm Planning_

Start    : `2026-06-01`


## Background and Motivation

We want to optimize how we purchase DNA sequences by considering the value of Degenerate-Codon Oligo Libraries. We can purchase a sequence where at specific positions, different bases can occur at a ratio we define. Using this, we could test many similar sequences from a petadex 60pid cluster even though we only purchased 1 sequence. Therefore, our motivation is to find the best sequence(s) to purchase for our needs, considering the following factors: 

1. **Sequence Quantity**: How many sequences are ordered.

    - Need some higher-level analysis to select which *families* to select from.
    - Could also run the algorithm on every family?? Depends on how expensive it is.

2. **Sequence Degeneracy**: How many degenerate bases we will allow in any given sequence.

    - Want to *minimize junk*, critical part of Artem's specification.

3. **Sequence Coverage**: How many natural amino acid sequences (exist in the chosen families) these purchased DNA sequences cover.

4. **RNA Structure**: Whether the RNA structure would be feasible (does it hairpin?)

    - Assign *-inf* to structures that aren't feasible. 
    - Computed using Claire's *scoring function*.

5. **Codon Ratios**: Which ratio of bases at these positions would maximize the amount of natural sequences synthesized and minimize the amount of artificial ones that aren't biologically existing.

    - Computed using Lisa's *scoring function*. This considers the codon table for specific vectors which assays would take place using.
    - Max 4 custom ratio bases according to Twist, although we may be able to pay for more
        - IUPAC Standard 50/50 combinations don't count towards this custom ratio count


## Problem Statement

We want to create an overall scoring function and algorithm that maximizes this function in order to choose the best sequences to order. 

## Section - Previous Solutions Summary

Claude Summary (*I've italicized my thoughts on the summaries and how they relate to our problem.*):

What the prior algorithms actually do:

SwiftLib (Jacobs et al. 2015) — DP, position-independent.
This is exactly the multiplicative/separable structure I described. DP state = smallest library achieving error e using positions 1..i; the recursion multiplies library sizes and sums per-position error. O(n²m²), runs in ~0.6s.

- *This is quite fast, should probably look into maybe just using this algo if its conditions are similar to ours.*

Its objective is not "minimize junk" directly — junk is controlled only by a hard cap on total library size L = ∏ᵢ |dᵢ|. Error = sum of omitted observed-AA counts per column.

Hard limitation, stated explicitly: it cannot model covariation. Optimizing residue-pair information is what makes it NP-complete, so they deliberately assume column independence. 
It also freely packs in unobserved AAs (junk) to economize on DNA. This is the price of separability.

- *Think about these limitations, we do need to minimize junk, but I think that column independence is fine? Not sure what we could do with modelling/taking advantage of the columns if they're dependent on each other.*

- *Note: This is useful for avoiding junk. If two positions throughout the library are highly correlated, it'd be better to encode for the difference using separate oligo sequences, rather than degeneracy which introduces 50% junk.*

DeCoDe (Sumida et al. 2020) — ILP, position-dependent.

This one tackles the non-separable, conjunctive coverage head-on. It has a variable X_ij = "target sequence i is fully encoded by template j" (all positions must match — your conjunction), and maximizes the count of covered targets subject to a log-library-size budget.

It already solves two of your "missing" sub-problems:
Non-constant length → gap codons. Templates can use gap codons to produce shorter proteins; demonstrated on avGFP (238 vs 239 aa). So length variation is handled by gaps in the template, not a separate mechanism.

- *I think these gaps could solve some of the issues with non-constant length, but there is pretty large variation in our families*

Multiple oligos → "sublibraries." Group similar sequences, one degenerate template each, combine under a shared budget. This is the partitioning idea — but the ILP finds the grouping implicitly, which is exactly why it's expensive (hours to days; 48h cap with optimality gaps; scales exponentially in #sublibraries).
On NP-hardness: DeCoDe claims it via reduction from protein design (Pierce & Winfree 2002), not from clique. Your "reduction from clique" may be a different/independent argument — worth pinning down which, since it affects which approximation results you can borrow. I didn't find a clique reduction in these papers.
Notably: DeCoDe names ancestral reconstruction as a target use case — but it does not use phylogeny in the algorithm. Templates are free ILP variables.

- *Expand on this*
- *Our phylogenetic tree idea doesn't need to find the grouping manually, since phylogenetic trees do it for us.*
- *Is this necessarily better? Not sure, will determine from the benchmarks but the argument that you get both a phylogenetic tree and the oligo design from the algorithm is a potentially desirable one.*

DeCOIL (Yang et al. 2023) — stochastic search, fitness-guided.

Different objective entirely: it optimizes DC libraries to sample high-fitness and diverse variants for ML-assisted engineering, scored by a learned fitness predictor, via direct stochastic search. Relevant to you only if/when you want a "predicted-good" weight rather than pure natural-sequence coverage — but it shows the third algorithmic family (neither DP nor ILP: score-guided local search).

- *interesting to hear, maybe we could pivot to/expand on this once we have a ML model that can better predict plastic degredation ability/metadata.*

## Section: Brainstorming - Functions

### 1. *Compute Coverage* Function: def compute_coverage(sequence, family) -> int

The main idea of this function is to see how many sequences can be covered by an oligo.

Can't compute all possible proteins and check if the $N$ real sequences of length $L$ are in this set - set grows exponentially and is predominantly trash (hopefully not but that's the next section). 

Instead, loop over the real sequences and then for each amino acid, search if it can be encoded by the codon at the corresponding position. 

Can also avoid more complex data structures and search logic by encoding the genetic code with degenerate codons:

1. Store a dictionary of the genetic code (one for either direction).
2. Can create a dictionary for all of the amino-acids encoded by every degenerate codon, as there are $15^3$ of them. This is computed using the result of step 1.
3. Create $A_{c}$ list, where index $i$ contains the set of amino acids that can be encoded by the degenerate codon at position i, using the result from step 2. This will be a length 20-bitmap, where amino acid $a$ will consistently be at bit position $i_a$.
4. Loop over positions:
    a. Use $A_{c}[i]$ for every position $i$.
    b, $(A_c[i] >> i_a) \& 1$: Shifts the bit that represents membership of the AA of interest to the right, returns 1 if the bit is 1.
5. Early return if any are impossible.


This will run in worst case $O(N * L)$.

Could be further optimized with some preprocessing work to determine if some columns aren't decision columns, and therefore don't need to be recomputed every time because there's no variance.

Also need to consider what to do with columns with gaps -- more discussion on this later.


### 2. *Minimize Junk* Function: def min_junk(degen_bases, ratios) -> float

There's 2 types of junk, Coverage Junk $J_c$ and Ratio Junk $J_r$.

Let $C \in \mathbb{N}$ represent the total sequence space encoded for by the degenerate codons. 

$C = \prod_{i} d_i$, where $d_i$ is the number of amino acids producible at codon position c. 

Let $N \in \mathbb{N}$ be the number of real sequences that are reached by the oligo sequence.

Intuitively: $J_c = P - N$.

**Note:** Minimizing coverage junk is equivalent to maximizing coverage, which is why it may not be useful as a junk metric within the algorithm. I am recording this though in case we ultimately want to choose which sequences from which libraries to choose, as we can't send a sequence for each library to wet lab. $J_c + J_r$ would be a good way to compare the solutions between libraries as a total junk cost.

To define $J_r$, 

We need to formalize this better:

F = {x⁽¹⁾,…,x⁽ᴹ⁾}, with c codon positions. A candidate oligo O assigns to each codon position c three base distributions (p_{c,1}, p_{c,2}, p_{c,3}) over {A,C,G,T}.

Per-position amino-acid distribution (this is where the genetic code enters):

$P_c(a) = Σ_{(b₁b₂b₃) : code(b₁b₂b₃)=a}  p_{c,1}(b₁)·p_{c,2}(b₂)·p_{c,3}(b₃)$



NOTE: Lisa's organism encoding score could also play into this.

#### Formalizing `min_junk`

**Setup.** Let the target family be a set of aligned amino-acid sequences

$$
\mathcal{F} = \{x^{(1)}, \dots, x^{(M)}\}, \qquad x^{(i)} = (x^{(i)}_1, \dots, x^{(i)}_C),
$$

over $C$ codon positions. A candidate degenerate oligo $O$ assigns to each codon position $c$ three independent base distributions over the alphabet $\{A,C,G,T\}$:

$$
\big(p_{c,1}, \; p_{c,2}, \; p_{c,3}\big), \qquad \sum_{b \in \{A,C,G,T\}} p_{c,j}(b) = 1 .
$$

**Per-position amino-acid distribution.** The genetic code enters here. The probability that position $c$ produces amino acid $a$ (or a stop) is

$$
P_c(a) \;=\; \sum_{\substack{(b_1 b_2 b_3) \\ \text{code}(b_1 b_2 b_3) = a}} p_{c,1}(b_1)\, p_{c,2}(b_2)\, p_{c,3}(b_3),
\qquad a \in \mathcal{A}_{20} \cup \{\text{stop}\}.
$$

Stop mass $P_c(\text{stop})$ is junk automatically (truncation) — this is why $\text{NNK}$ is preferred over $\text{NNN}$.

**Synthesis distribution (always a product).** Mixed-base synthesis draws each position independently, so the distribution over produced proteins factorizes:

$$
Q_O(x) \;=\; \prod_{c=1}^{C} P_c(x_c).
$$

**Coverage mass** (covariation-aware — the product is summed *only over real target members*):

$$
\mathrm{Cov}(O) \;=\; \sum_{x \in \mathcal{F}} Q_O(x) \;=\; \sum_{x \in \mathcal{F}} \prod_{c=1}^{C} P_c(x_c).
$$

**Junk objective.**

$$
\boxed{\; J(O) \;=\; 1 - \mathrm{Cov}(O) \;=\; 1 - \sum_{x \in \mathcal{F}} \prod_{c=1}^{C} P_c(x_c) \;\in\; [0,1]. \;}
$$

A single number that absorbs every junk source: stop codons, cross-product **covariation leakage**, and mass misallocated to rare variants.

---

#### Division of labor

Because $Q_O$ is forced to be a **product** distribution, it can match only the *marginal* per-position frequencies — never correlations. Hence:

- **Custom ratios** $\Rightarrow$ remove *marginal* junk (mass bled onto rare variants at a single position).
- **Splitting into multiple oligos** $\Rightarrow$ remove *covariation* junk (e.g. the $\{VT, IS\}$ cross-product that also produces $VS, IT$).

#### Twist custom-ratio budget

Unlimited IUPAC positions (constrained to **uniform** mixes over a base subset: $R, V, N, \dots$) plus at most **4 custom** positions with arbitrary continuous ratios. 

Idea: Use greedy allocation and spend the 4 custom postiions on the positions whose family marginal is *most skewed*.

$$
\Delta_c \;=\; J_{\text{IUPAC}}(c) - J_{\text{custom}}(c), \qquad \text{pick } \arg\text{top-4}\; \Delta_c .
$$

#### Caveat: minimize subject to fixed support

Minimizing $J$ alone is degenerate — collapsing every position to one amino acid gives $\mathrm{Cov}=1$ but covers a single sequence. So $J$ is minimized **subject to a fixed support** $S_c = \{a : P_c(a) > 0\}$ set by the coverage step, with $p_{c,j}(b) > 0$ kept for every chosen base. With the support fixed the objective is interior; coordinate ascent over the $\le 4$ small simplices converges quickly.

```python
def junk_fraction(oligo, family) -> float:
    """J(O) = 1 - sum_{x in family} prod_c P_c(x_c).  Pure evaluation."""

def optimize_ratios(structure, family, custom_budget=4) -> tuple["Ratios", float]:
    """Given fixed support, choose <=4 custom-ratio positions + continuous
    ratios minimizing junk_fraction. Returns (ratios, achieved_junk)."""
```


### 3. How to deal with Families where there is non-constant sequence length:


*Baseline: Huge issue since the oligo can't synthesize sequences of different lengths.*




 

### 4. *Phylogenetic Tree* Approximation (DP):

This approach will use the phylogenetic tree as a starting point.
This is useful since each node has two children, allowing for propagation of solutions through
a *DP* array using the following Bellman equation. 

Let $DP[r]$ represent the optimal score of the objective function and the corresponding sequence for this solution for a tree rooted at node $r$. Note that the root node r has two children in the phylogenetic tree, $L$ and $R$. 

Define *introduce_degen(L, R) -> sequence* to be the function to introduce degeneracy to create a sequence which can produce both of its children's sequences.

$DP[r][0] = max[DP[L][0], DP[R][0], \text{scoring\_func}(\text{introduce\_degen}(L, R))]$

$DP[r][1] =$ argmax sequence from the above equation.


Also needs a budget column, think about how to implement this.

### 5. *Phylogenetic Tree + Ancestral Reconstruction* Approximation (Greedy):

Old Planning Notes:

This approach will traverse the ancestral reconstruction tree from the root outwards, adding degeneracy in the most opportune positions using the reconstructions as a base.

Similar to greedy algo in essence, this won't actually get the best coverage but it'll get a pretty good ancestral reconstruction - based oligo which could be fine for a first pass?

Talk more with Thomas about this.

# Algorithm First Pass - July 1st Deadline

After preprocessing is done, we have the aligned AA sequences and we want to design a good greedy algorithm to solve this problem.

#### Revised preprocessing plan: 
Verification (cluster-agnostic): envelope trim via per-cluster pyhmmer HMM → foldability certificate via ESMFold/ESMC pLDDT. No constant-length assumption anywhere.
cluster1 fix: its spread genuinely is terminal-tag artifact (I confirmed the internal anchor span is invariant here), so trimming to the envelope is right — but I'll certify it by envelope + foldability, not by forcing zero spread, so the same code works on the next cluster where ±5% is real.

### First Prototype:

This first prototype of the algorithm was implemented to handle the following:
1. Greedy choice implementation for degenerate codon insertion (fragments can come later)
2. Converting between nucleotide bases and amino acids

The following simplifying assumptions are made and will be laxed over the development of the sophisticated algorithm:
1. Preprocessing has been completed and we are running the algorithm on a selected subset of the processed orfs. We want to run on the subset where all sequences are the same length, and this is the largest subset of this sort. This should be an ok approximation for now

Therefore this algorithm takes this cluster1.core.fasta and will return the best single oligo with degeneracy for this cluster.

## Degenerate-Oligo Design — Algorithm (current state - Jul 29th, 2026)

**Preprocessing (per cluster)**

1. **Align** — MSA of the cluster's cores (MAFFT).

2. **Trim** — cut to the catalytic domain; drop fragments + dedup (various occupancy/length methods; loop-binning for indels).
- Cores are NOT all equal length.

*Two other methods for this to try next*:
a. Using HMM for this trimming, move out from the HMM coordinates until conservation stops.
b. Fold the centroid then extract the catalytic domain from that.

**Design (prototype, `greedy_oligo.py`)**

3. **Largest equal-length subset** — design only over the modal-length cores so every position is a clean column (sidesteps indels; other lengths dropped).

4. **Budget** — hard cap on # degenerate nucleotide positions (default 10).

*To try next:*
a. More sophisticated methods of accounting for budget (degen codons + fragment ordering cost)

5. **Greedy set-cover** — start from a seed core (free, 0 bases); repeatedly add the not-yet-covered core that maximizes *(weighted sequences newly covered) ÷ (extra degenerate bases)*, while staying ≤ budget. 

Multi-start over all seeds, keep the best run. A core is "covered" only if EVERY residue is encoded.

6. **Output** — concatenate per-column codons into the oligo; report degenerate bases used, library size (∏ codon degeneracies = junk), and weighted coverage.


*Known gaps:* single oligo only (multi-oligo / K>1 not yet); equal-length subset drops the rest of the cluster; greedy is approximate, not provably optimal.

# Fragment Idea:

1. Older Papers

- SCHEMA (the "broken contacts" disruption score): Voigt, Martinez, Wang, Mayo, Arnold, "Protein building blocks preserved by recombination," Nat Struct Biol 2002. The library-analysis follow-up with the worked math: Meyer et al., Protein Science 2003 (PMC2323955).

- RASPP (crossover selection as shortest path): Endelman, Silberg, Wang, Arnold, "Site-directed protein recombination as a shortest-path problem," PEDS 2004 (PubMed). Reference implementation: github.com/mattasmith/SCHEMA-RASPP. Protocol write-up: Designing Libraries of Chimeric Proteins Using SCHEMA Recombination and RASPP.

- Haplotype-block DP: Zhang et al., "A dynamic programming algorithm for haplotype block partitioning," PNAS 2002 (PMC124231), and the empirical block concept in Gabriel et al. 2002. Tag-SNP/DP software: HapBlock.

Good new one: https://onlinelibrary.wiley.com/doi/full/10.1002/pro.5169



# Fragment Implementation — First Pass Summary (`fragment_split.py`, Jun 29 2026)

This is the first working version of the **fragment** design (separate from the single-oligo `greedy_oligo.py`). Instead of one degenerate oligo across the whole core, it cuts the core into **3 physically-synthesized fragments** that are assembled combinatorially (Golden Gate / homologous recombination). Code: `resources/260629_issue84_algoplanning/fragment_split.py`.

## 1. How to run it (in simple terms)

The pipeline is three steps:

1. **Preprocess → cores** (already done): the per-cluster `*.core.fasta` (deduped catalytic cores, headers carry `_nK` natural-sequence counts as weights).
2. **Align the cores** so positions line up into columns (gaps become `-`):
   ```bash
   mafft --auto cluster1.core.fasta > cluster1.core.aln.fasta   # in WSL
   ```
3. **Run the designer** (`python3`):
   ```bash
   python3 fragment_split.py cluster1.core.aln.fasta --junk-budget 1000
   ```

Every run is auto-saved to `algoruns/<timestamp>_<cluster>_<chemistry>/` containing `report.txt`, `summary.json`, and `fragment[1-3].fasta` (the actual pieces you'd order). Key flags: `--junk-budget` (hard cap on library size — the main knob), `--chemistry {agnostic,gg,hr}` (cut-site rule; agnostic = no assumption about where cuts can go), `--max-degenerate` (optional oligo compression, default 0).

## 2. The DP formulation — and why we approximate it with 3 fragments

**General problem (the real DP).** Choosing where to cut is a partition of the alignment columns into contiguous fragments. Model it as a graph: **nodes = column boundaries (candidate cut sites), edges = fragments (a column range)**. A full design = a path from the start boundary to the end boundary; each edge has a cost (cost to encode that fragment's diversity). The optimal cut set = the **shortest path** through this graph — this is the *Recombination-As-Shortest-Path (RASPP)* / *SwiftLib-DP* shape. The Bellman recursion:

$$
f(j, m) \;=\; \min_{i < j}\;\Big[\, f(i, m-1) \;+\; \mathrm{cost}\big(\text{fragment } i{\to}j\big) \,\Big],
$$

where $f(j,m)$ = best cost to cover columns $1..j$ using $m$ fragments, with fragment-length bounds. Recover the cuts by backpointers. Complexity $O(L^2 K)$ for $K$ fragments.

**Why 3 for now.** We cap at **K = 3 fragments → exactly 2 cuts**, for two reasons: (a) more fragments = more assembly junctions = more recombination junk and lower assembly yield, so 3 is a sensible wet-lab ceiling for a first pass; (b) with only 2 cuts we don't even need the DP — we **brute-force enumerate every legal pair of cut sites** ($O(L^2)$), which is the K=3 special case of the recursion above. When we later want K > 3, the same edge/cost model drops straight into the Bellman DP — no rework.

**What the budget means here.** Mirroring `greedy_oligo`'s degenerate-base budget, the fragment version uses a **hard junk budget = max producible library size**. Given the 2 cuts, a *global* greedy grows one consistent set of whole cores to cover and **drops** the cores that don't fit under the cap. A core whose 3 pieces are all already present is a **free recombinant** (zero added junk) — that's the combinatorial payoff. (Dropping must be global, not per-fragment: coverage is conjunctive — a core only exists if *all 3* of its pieces are in the library.)

## 3. Results over varying budgets — how much can we cover cheaply?

Run on the two iSPETase clusters at increasing junk budgets (coverage = fraction of **natural sequences**, weighted by `_nK` counts):

| cluster | budget 100 | budget 1,000 | budget 10,000 |
|---|---|---|---|
| **cluster1** (ragged ends, flat weights) | 36% — 23 oligos | 85% — 65 oligos | 100% — 85 oligos |
| **cluster2** (real C-terminal loop, weight-concentrated) | **87% — 22 oligos** | 100% — 42 oligos | 100% |

**Conclusions:**

- **Cheap budgets already give a meaningful return when sequence weight is concentrated.** cluster2 covers **87% of all natural sequences with only ~22 short oligos and ≤100 library members** — because the greedy covers the high-count cores first and lets shared pieces recombine for free. This is the "modest cost, big payoff" regime we wanted.
- **Fragments help most where there's real modular structure.** cluster2 (a genuine loop indel + concentrated weights) is a strong fragment candidate; cluster1 (mostly ragged-terminus artifact, flat weights) is weak (36% at the same budget) — **cluster1 is better served by a single oligo** (`greedy_oligo`).
- **The junk budget is honestly enforced** every run (library ≤ budget), and **length variation is handled for free** — a different-length piece is just another variant in that fragment layer, since cuts land in conserved columns and the assembly only cares about fragment *ends*, not interior length.
- Both clusters have a perfectly **conserved middle band** that the cutter finds on its own — a natural overhang/anchor site.

## 4. What's next — and how it maps to GGAssembler

The single most useful paper for this direction is **GGAssembler** (Hoch & Fleishman, *Protein Science* 2024, [doi:10.1002/pro.5169](https://onlinelibrary.wiley.com/doi/full/10.1002/pro.5169); code: `Fleishman-Lab/GGAssembler`). Our design is the *same graph* as theirs (nodes = cleavage points, edges = fragments, shortest path over edges, edge cost = nucleotides to encode the fragment's diversity). The key relationship:

> **GGAssembler is the intended BACK END; our script is the FRONT END.** GGAssembler takes the *desired diversity as given* (a resfile: which residues at which positions) and only minimizes DNA \$ — it does **not** decide *what* to put in each fragment and does **not** model covariation. Our script *derives* the diversity from the natural cluster and adds the junk-budget / coverage objective. So the clean division is: **we choose cuts + which cores/pieces to keep → hand the kept pieces to GGAssembler → it returns the cheapest oligos + orthogonal overhangs.**

Concrete next steps, in order:

1. **Make cut-selection budget-aware.** Right now Phase A picks the 2 cuts by minimizing the all-core distinct-piece product (a covariation proxy), *blind* to the junk budget and the dropping. It should score cuts by the coverage actually achievable under the budget.
2. **Better linkage signal for cut placement.** Use APC-corrected mutual information or DCA-style coupling (not raw MI — our clusters are low-diversity and phylogenetically confounded) to cut at "linkage valleys," keeping co-varying columns inside one fragment. (Note: ESMFold is **not** needed for cut placement — covariation stats suffice; structure only matters for the separate foldability/junk filter.)
3. **Adopt GGAssembler's overhang model directly.** Its **"rainbow shortest path"** enforces that the Golden Gate overhangs chosen along the path are mutually orthogonal / high-fidelity (a hard, ~20–30 limited resource). That's exactly our pluggable `--chemistry gg` hook — we should hand junctions to GGAssembler rather than reinvent overhang selection. (For HR/Gibson the overhang budget effectively disappears but cut sites need longer conserved anchors — our `--chemistry hr` stub.)
4. **Generalize K = 3 → K via the Bellman DP** once the 3-fragment results justify more junctions.

*Reading priority for the team:* GGAssembler (back end + graph model) → DeCoDe 2020 (the mixed-length / linked-variation objective done exactly, but NP-hard) → RASPP 2004 (the shortest-path cut-selection foundation).

# Tuesday Meeting Notes:

Include fold coverage [#72](https://github.com/petadex/igem-toronto/issues/72)

\# of wildtype

Hard cutoff on the % of the junk

Don't limit the number of fragments

Assume that the preprocessing is gonna be HMMs and don't worry about it

# Final Algorithm - `marginal_design.py` (Jul 1 2026)

Successor to `fragment_split.py`. It implements the three decisions from the Tuesday meeting notes above (**hard cutoff on the % of junk**, **don't limit the number of fragments**, **assume HMM preprocessing**) and makes Golden Gate / Gibson-HR validity **self-contained**, so no downstream tool is needed to check that a design is buildable. Code: `resources/260629_issue84_algoplanning/marginal_design.py`.

## 1. What changed vs. `fragment_split.py` (and why)

| change | rationale |
|---|---|
| K fixed at 3 -> **any K = 1..12** (RASPP/SwiftLib DP places the K-1 cuts; the tool sweeps all K and prints a frontier) | Tuesday decision *"don't limit the number of fragments"*; wet lab realistically supports 1-12 junctions. Lets us actually **test** whether few fragments is best (it is - see results). |
| hard library-size budget -> **junk as a % of the produced library** (`--max-junk-pct`, default 80) | Tuesday decision *"hard cutoff on the % of the junk"*. A fraction is comparable across designs and far easier to reason about than an absolute library size. It acts as a feasibility constraint: greedily add cores by cheapest junk-per-new-sequence **while** the design's junk fraction stays <= cutoff; stop when no core fits. |
| recommends by **sequences-per-oligo** ("a few extra full-length seqs per order"), not max coverage | team philosophy *"smaller gains for less junk"*. Also, maximizing coverage trivially returns K=1 (synthesize every gene = 0% junk by construction), so it was useless as a recommender. |
| reports the concrete headline number: **X/N unique cores, W/T natural sequences encoded** | requested: always state how much of the cluster the encoding actually produces. |
| `--chemistry gg`/`hr` are now **real and self-contained**, not placeholders | see section 3 - the whole point of making the algorithm not depend on GGAssembler. |
| degenerate codons are independent of K and applied within **any** fragment (incl. K=1) via junk-gated `--degenerate` | they compress equal-length pieces into fewer oligos, but only when it adds **zero** junk (so degenerate codons can't silently blow up the library). |

## 2. How to run

```bash
python marginal_design.py ../../ninetypidorfs\cluster2.core.aln.fasta --chemistry gg
python marginal_design.py ../../ninetypidorfs\cluster2.core.aln.fasta --chemistry gg --max-junk-pct 50
```
Key flags: `--chemistry {agnostic,gg,hr}`, `--max-junk-pct` (the junk cutoff, main knob), `--k-max` (default 12), `--min-block-cols`, `--degenerate`, `--arm-codons` (hr). Every run is saved to `algoruns/<ts>_<cluster>_<chem>_marginal/` (`report.txt`, `summary.json` with the validated junction overhangs/arms, `fragment*.fasta`).

## 3. Golden Gate / HR validity is now IN the algorithm (no GGAssembler needed to validate)

The old plan called GGAssembler "the back end" we would hand fragments to. That was wrong: **GGAssembler chooses its own cuts**, so you cannot pass it fixed fragments for it to "improve/validate" - it overlaps our job rather than sitting after it, and if it re-derived cuts it would throw away our covariation-aware placement and junk minimization. So instead we fold the two levels of assembly validity directly into cut placement:

- **Level 1 (per site, a per-edge property).** `gg`: a 4-nt BsaI overhang must sit in immutable (constant) flanking codons and be individually high-fidelity (non-palindromic, balanced GC). Because we synthesize the constant fragments, a site offers a *set* of achievable overhangs via synonymous codons. `hr`: a constant homology-arm window of `>= --arm-codons` residues must straddle the cut.
- **Level 2 (per set - the part that is NOT a per-edge property).** `gg`: the chosen overhangs must be mutually orthogonal (not identical, and not within 1 mismatch of each other's reverse complement - the single-mismatch ligation that dominates GGA infidelity). This is exactly GGAssembler's "rainbow" constraint. We enforce it **during** an exact DFS that carries the chosen overhangs along the path, so it is impossible for the tool to emit a cross-reacting set. `hr`: homology arms must be mutually distinct (near-identical arms mis-recombine).

Fidelity is a transparent, literature-grounded **model** (Potapov 2018 / Pryor 2020: <=1-mismatch ligation dominates infidelity, palindromes self-ligate, extreme-GC overhangs ligate poorly). The exact empirical NEB 256x256 ligation table can be dropped into `_overhang_ok`/`_gg_conflict` without touching the algorithm.

> **Result: every `gg`/`hr` design the tool returns is buildable as-is** - each junction carries a validated, individually high-fidelity overhang/arm, and the whole set is mutually orthogonal/distinct by construction. GGAssembler is now **optional** (only to squeeze codon cost further via DLX exact-cover); it is **not required for correctness**.

## 4. Results (both iSPETase clusters, `--max-junk-pct 80`)

**Recommended designs** (max sequences-per-oligo). Coverage is natural-sequence (weighted) coverage; unique cores in parentheses.

| cluster | chem | K | coverage (nat seqs) | cores | junk fraction | oligos |
|---|---|---|---|---|---|---|
| **cluster2** (strong) | gg | 4 | **122/153 = 80%** | 13/41 | 75.0% | 18 |
| **cluster2** (strong) | hr | 5 | **117/153 = 76%** | 10/41 | 79.2% | 12 |
| **cluster1** (weak) | gg | 2 | **41/66 = 62%** | 37/62 | 77.6% | 38 |
| **cluster1** (weak) | hr | 2 | **40/66 = 61%** | 36/62 | 77.5% | 37 |

**The key insight about junk fraction:** it sits **pinned just under the 80% cutoff in essentially every design** - the marginal selection fills coverage up to the cap. So junk fraction is set by `--max-junk-pct`, and is **not** the differentiator between chemistries or K. The real trade-off the frontier exposes is **coverage vs. number of oligos, at (near-)constant junk**:

| cluster2 `gg` | K | coverage | junk | oligos |
|---|---|---|---|---|
| max coverage | 2 | 152/153 = **99%** | 79.2% | 38 |
| balanced | 3 | 146/153 = 95% | 79.2% | 35 |
| **recommended (cheapest)** | 4 | 122/153 = 80% | 75.0% | **18** |

i.e. at the same ~79% junk you can buy **99% coverage for 38 oligos** or **80% for 18 oligos**. `hr` behaves the same (K=2: 152/153 @ 38 oligos -> K=5: 117/153 @ 12 oligos). Example validated overhangs (cluster2 `gg` K=4): `CCAC / CAAG / GGAC` - all balanced-GC, non-palindromic, mutually orthogonal; `CAAG` is one of the overhangs in the GGAssembler paper's own figure.

**Conclusions:**
- **cluster2 is genuinely fragmentable** under both chemistries - it keeps ~80% of natural sequences while roughly **halving** the oligo count vs. plain gene synthesis (K=1 = 41 oligos). `gg` and `hr` land in the same ballpark; `hr` a bit cheaper (longer arms allow different cuts), `gg` a bit higher coverage per K.
- **cluster1 is not fragmentable** - both chemistries top out at ~61% coverage for ~37 oligos, barely better than the 62-oligo K=1 baseline. Confirms the earlier `fragment_split` finding: cluster1 is better served by a single oligo / direct gene synthesis.
- Want **less junk**? Lower `--max-junk-pct` and every design re-solves under the tighter cap (fewer sequences, fewer oligos). e.g. cluster2 `gg` @ 50%: K=2 gives 121/153 with 9 oligos.

## 5. Known gaps / next

- Fidelity is a **model**, not the exact NEB empirical matrix - swap-in ready in `_overhang_ok`/`_gg_conflict`. Could also adopt a curated high-fidelity overhang set (Pryor 2020) directly.
- `gg` DFS on large/constant-dense clusters (e.g. cluster1) takes tens of seconds per full K-sweep (node budget 3M); the `agnostic` DP is instant.
- Still coverage + covariation + assembly-validity only. Not yet integrated: **ratio junk** (J_r) / Twist custom ratios and the **RNA-structure/foldability** filter from the Functions brainstorm above.